In [ ]:
import os, time, random, json
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
import pandas as pd


# Reproducibility

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[setup] device = {device}")


# Data

train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])  # -> [-1,1]
])
test_tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

data_root = "./data"
train_ds = datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tfms)
test_ds = datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_tfms)

# Use a *consistent* train/val split for all experiments
val_fraction = 0.1
val_size = int(len(train_ds) * val_fraction)
train_size = len(train_ds) - val_size
train_ds, val_ds = random_split(train_ds, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

def make_loaders(batch_size: int) -> Tuple[DataLoader, DataLoader, DataLoader]:
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    )


# Models

class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        w = self.pool(x)
        w = self.fc(w)
        return x * w

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, use_se=False):
        super().__init__()
        p = k // 2
        self.conv = nn.Conv2d(in_ch, out_ch, k, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.se = SEBlock(out_ch) if use_se else nn.Identity()
    def forward(self, x):
        x = self.act(self.bn(self.conv(x)))
        return self.se(x)

class Architecture1(nn.Module):
    # A small, reasonably strong baseline. We hold this fixed in Stage A (optimizer sweep).
    def __init__(self, num_classes=10):
        super().__init__()
        self.body = nn.Sequential(
            ConvBlock(3, 64), ConvBlock(64, 64), nn.MaxPool2d(2), nn.Dropout2d(0.1),
            ConvBlock(64, 128), ConvBlock(128, 128), nn.MaxPool2d(2), nn.Dropout2d(0.2),
            ConvBlock(128, 256), nn.MaxPool2d(2), nn.Dropout2d(0.3),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.body(x)
        return self.head(x)

class Architecture2(nn.Module):
    # Architectural variant for Stage B.
    def __init__(self, num_classes=10, use_se=False):
        super().__init__()
        self.body = nn.Sequential(
            ConvBlock(3, 96, use_se=use_se), ConvBlock(96, 96, use_se=use_se), nn.MaxPool2d(2), nn.Dropout2d(0.15),
            ConvBlock(96, 192, use_se=use_se), ConvBlock(192, 192, use_se=use_se), nn.MaxPool2d(2), nn.Dropout2d(0.25),
            ConvBlock(192, 256, use_se=use_se), nn.MaxPool2d(2), nn.Dropout2d(0.35),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(self.body(x))

#  loss 
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes=10, smoothing=0.1):
        super().__init__()
        self.conf = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
    def forward(self, pred, target):
        pred = F.log_softmax(pred, dim=1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.unsqueeze(1), self.conf)
        return torch.mean(torch.sum(-true_dist * pred, dim=1))


# Training 

@dataclass
class TrainConfig:
    epochs: int = 20
    batch_size: int = 128
    lr: float = 3e-4
    weight_decay: float = 5e-4
    optimizer_name: str = "adamw"
    label_smoothing: float = 0.0

def make_optimizer(params, name: str, lr: float, wd: float):
    name = name.lower()
    if name == "sgd":
        return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd, nesterov=True)
    if name == "adamw":
        return optim.AdamW(params, lr=lr, weight_decay=wd)
    if name == "rmsprop":
        return optim.RMSprop(params, lr=lr, weight_decay=wd, momentum=0.9)
    raise ValueError(f"Unknown optimizer {name}")

def train_one(model, loaders, cfg: TrainConfig):
    train_loader, val_loader, _ = loaders
    model.to(device)
    criterion = (LabelSmoothingLoss(smoothing=cfg.label_smoothing)
                 if cfg.label_smoothing > 0 else nn.CrossEntropyLoss())
    opt = make_optimizer(model.parameters(), cfg.optimizer_name, cfg.lr, cfg.weight_decay)
    best_val = -1.0
    best_state = None

    for epoch in range(1, cfg.epochs+1):
        model.train()
        running = 0.0
        correct = 0
        total = 0
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item() * x.size(0)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += x.size(0)
        train_loss = running / total
        train_acc = correct / total * 100

        # validation
        model.eval()
        vcorrect, vtotal, vloss_sum = 0, 0, 0.0
        with torch.no_grad():
            for x,y in val_loader:
                x,y = x.to(device), y.to(device)
                out = model(x)
                loss = F.cross_entropy(out, y)
                vloss_sum += loss.item() * x.size(0)
                vcorrect += (out.argmax(1) == y).sum().item()
                vtotal += x.size(0)
        val_acc = vcorrect / vtotal * 100
        val_loss = vloss_sum / vtotal
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1:
            print(f"[{cfg.optimizer_name:6s}] epoch {epoch:02d} | "
                  f"train {train_acc:5.1f}%/{train_loss:.3f} | val {val_acc:5.1f}%/{val_loss:.3f}")
    model.load_state_dict(best_state)
    return best_val

def evaluate(model, loaders):
    _, _, test_loader = loaders
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return correct / total * 100


# hyperparameter 


def tiny_grid_tune(model_ctor, loaders, optimizer_name: str, base_cfg: TrainConfig):
    grid = [
        (3e-4, 5e-4),
        (5e-4, 5e-4),
        (1e-3, 1e-4),
    ]
    results = []
    for lr, wd in grid:
        cfg = TrainConfig(**asdict(base_cfg))
        cfg.lr = lr
        cfg.weight_decay = wd
        cfg.optimizer_name = optimizer_name
        model = model_ctor()
        val = train_one(model, loaders, cfg)
        results.append((val, lr, wd))
    results.sort(reverse=True, key=lambda t: t[0])
    best_val, best_lr, best_wd = results[0]
    print(f"[tuning:{optimizer_name}] best val={best_val:.2f}% with lr={best_lr} wd={best_wd}")
    return best_lr, best_wd, results


# Stage A - optimiser

print("\\n=== Stage A: Optimiser sweep (single factor) ===")
base_cfg = TrainConfig(epochs=15, batch_size=128)  # short but informative
loaders = make_loaders(base_cfg.batch_size)

opt_candidates = ["sgd", "adamw", "rmsprop"]
stageA_rows = []
for opt_name in opt_candidates:
    best_lr, best_wd, _ = tiny_grid_tune(Architecture1, loaders, opt_name, base_cfg)
    cfg = TrainConfig(**asdict(base_cfg))
    cfg.optimizer_name, cfg.lr, cfg.weight_decay = opt_name, best_lr, best_wd
    model = Architecture1()
    val = train_one(model, loaders, cfg)
    test = evaluate(model, loaders)
    stageA_rows.append({
        "stage": "A",
        "aspect": "optimizer_only",
        "optimizer": opt_name,
        "lr": best_lr,
        "weight_decay": best_wd,
        "val_acc": round(val, 2),
        "test_acc": round(test, 2),
        "arch": "architecture 1"
    })

stageA_df = pd.DataFrame(stageA_rows)
print("\\n[Stage A] summary (val→test):")
print(stageA_df.sort_values("val_acc", ascending=False).to_string(index=False))


In [ ]:

# Stage B —  architecture


print("\\n=== Stage B: Architecture sweep (single factor) ===")
best_opt_row = stageA_df.sort_values("val_acc", ascending=False).iloc[0]
fixed_opt = best_opt_row["optimizer"]
fixed_lr = float(best_opt_row["lr"])
fixed_wd = float(best_opt_row["weight_decay"])
archs = [
    ("architecture 1", lambda: Architecture1()),
    ("architecture 2", lambda: Architecture2(use_se=False)),
    ("architecture 3", lambda: Architecture2(use_se=True)),  
]
stageB_rows = []
cfgB = TrainConfig(epochs=20, batch_size=128, lr=fixed_lr, weight_decay=fixed_wd, optimizer_name=fixed_opt)
for name, ctor in archs:
    model = ctor()
    val = train_one(model, loaders, cfgB)
    test = evaluate(model, loaders)
    stageB_rows.append({
        "stage": "B",
        "aspect": "architecture_only",
        "optimizer": fixed_opt,
        "lr": fixed_lr,
        "weight_decay": fixed_wd,
        "val_acc": round(val, 2),
        "test_acc": round(test, 2),
        "arch": name
    })
stageB_df = pd.DataFrame(stageB_rows)
print("\\n[Stage B] summary (val→test):")
print(stageB_df.sort_values("val_acc", ascending=False).to_string(index=False))

In [ ]:
# Stage C — optimiser + architecture + advanced layers


print("\\n=== Stage C: Multi‑aspect (opt + arch + advanced) ===")
top2_opts = stageA_df.sort_values("val_acc", ascending=False)["optimizer"].unique()[:2]
top2_arch = stageB_df.sort_values("val_acc", ascending=False)["arch"].unique()[:2]

stageC_rows = []
for opt_name in top2_opts:
    # retune lightly since optimiser changed
    best_lr, best_wd, _ = tiny_grid_tune(lambda: Architecture1(), loaders, opt_name, base_cfg)
    for arch_name in top2_arch:
        ctor = (lambda: Architecture1()) if arch_name == "architecture 1" else (lambda: Architecture2(use_se=("3" in arch_name)))
        for ls in [0.0, 0.1]:
            cfgC = TrainConfig(epochs=25, batch_size=128, lr=best_lr, weight_decay=best_wd,
                               optimizer_name=opt_name, label_smoothing=ls)
            model = ctor()
            val = train_one(model, loaders, cfgC)
            test = evaluate(model, loaders)
            stageC_rows.append({
                "stage": "C",
                "aspect": "multi_aspect (opt+arch+advanced)",
                "optimizer": opt_name,
                "lr": best_lr,
                "weight_decay": best_wd,
                "label_smoothing": ls,
                "val_acc": round(val, 2),
                "test_acc": round(test, 2),
                "arch": arch_name
            })
stageC_df = pd.DataFrame(stageC_rows)
print("\\n[Stage C] summary (val→test):")
print(stageC_df.sort_values("val_acc", ascending=False).to_string(index=False))



out = {
    "stageA": stageA_rows,
    "stageB": stageB_rows,
    "stageC": stageC_rows,
    "seed": SEED,
    "device": str(device),
}
os.makedirs("results", exist_ok=True)
with open("results/experiment_summary.json", "w") as f:
    json.dump(out, f, indent=2)
stageA_df.to_csv("results/stageA.csv", index=False)
stageB_df.to_csv("results/stageB.csv", index=False)
stageC_df.to_csv("results/stageC.csv", index=False)
print("\\nSaved CSVs to results/ and a compact JSON summary.")

# Quick final take (one line): choose the best by validation from Stage C.
all_df = pd.concat([stageA_df, stageB_df, stageC_df], ignore_index=True, sort=False)
best_row = all_df.sort_values("val_acc", ascending=False).iloc[0]
print(f"\\nBest-by-val overall → arch={best_row['arch']}, opt={best_row['optimizer']},"
      f" lr={best_row['lr']}, wd={best_row['weight_decay']}, "
      f"label_smoothing={best_row.get('label_smoothing', 0.0)}, "
      f"val={best_row['val_acc']}%, test={best_row['test_acc']}%")


